<a href="https://colab.research.google.com/github/IBM/vLLM-Hook/blob/main/notebooks/demo_spotlight_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight Your Instructions: Instruction-following with Dynamic Attention Steering

vLLM-Hook is an extensible framework that allows selective access to model internals during inference. This notebook demonstrates **Spotlight**, an inference-time attention steering method that nudges attention toward emphasized instruction spans.

**Paper**: [Venakteswaran and Contractor, EACL 2026](https://aclanthology.org/2026.eacl-long.174/)

Spotlight requires eager execution so the hook can access attention tensors during prefill.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo, installs CUDA 12.1-compatible runtime dependencies, installs `vllm_hook_plugins`, and refreshes the current Python process so the following cells can run without a manual runtime restart.


In [1]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
import importlib
import importlib.metadata as importlib_metadata
import os
import re
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/IBM/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("VLLM_HOOK_REPO_DIR", "/content/vLLM-Hook"))
PLUGIN_SRC = REPO_DIR / "vllm_hook_plugins"

# These must be set before vLLM is imported for the first time in this kernel.
os.environ["VLLM_USE_V1"] = "1"
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "fork")
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")


def run(cmd, cwd=None, check=True):
    print("+ " + " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout, end="")
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout)
    return result


def normalized_package_name(requirement_line):
    line = requirement_line.split("#", 1)[0].strip()
    if not line or line.startswith("-"):
        return ""
    return re.split(r"[<>=!~;\[]", line, maxsplit=1)[0].strip().lower().replace("_", "-")


def version_tuple(version):
    parts = []
    for part in re.split(r"[.+-]", version):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def protobuf_needs_pin():
    try:
        current = importlib_metadata.version("protobuf")
    except importlib_metadata.PackageNotFoundError:
        return True
    parsed = version_tuple(current)
    return not ((5, 29, 6) <= parsed < (6, 30))


def fail_if_already_imported(package_names):
    imported = sorted(
        name for name in package_names
        if name in sys.modules or any(mod.startswith(name + ".") for mod in sys.modules)
    )
    if imported:
        raise RuntimeError(
            "The setup cell needs to run before importing "
            + ", ".join(imported)
            + ". Restart the runtime once, then use Runtime > Run all."
        )


if IN_COLAB:
    fail_if_already_imported(["torch", "torchvision", "torchaudio", "vllm"])

    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
    else:
        run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH])

    filtered_req = Path("/tmp/vllm_hook_colab_requirements.txt")
    req = REPO_DIR / "requirement.txt"
    if req.exists():
        keep = []
        skip_packages = {"vllm", "torch", "torchvision", "torchaudio", "protobuf", "pillow", "pil"}
        for line in req.read_text(encoding="utf-8").splitlines():
            if normalized_package_name(line) in skip_packages:
                continue
            keep.append(line)
        filtered_req.write_text("\n".join(keep) + "\n", encoding="utf-8")
        run([sys.executable, "-m", "pip", "install", "-r", filtered_req])

    if protobuf_needs_pin():
        if "google.protobuf" in sys.modules:
            raise RuntimeError(
                "Colab has already imported google.protobuf, but its installed protobuf "
                "version is outside the range needed by this notebook. Restart the runtime "
                "once, then run this setup cell before any other imports."
            )
        run([sys.executable, "-m", "pip", "install", "--upgrade", "protobuf>=5.29.6,<6.30"])

    run([sys.executable, "-m", "pip", "install", "-U", "uv"])

    # Current Colab GPU runtimes ship CUDA 12.8-era PyTorch wheels. Installing
    # torch and vLLM in one transaction keeps their ABI pins aligned and avoids
    # the old CUDA 13 resolver path without requiring a runtime restart.
    vllm_install_base = [
        sys.executable, "-m", "uv", "pip", "install",
        "--system", "--reinstall", "--no-cache", "--torch-backend=cu128",
        "torch", "torchvision", "torchaudio",
    ]
    exact_vllm = run(vllm_install_base + ["vllm==0.19.0"], check=False)
    if exact_vllm.returncode:
        print("vLLM 0.19.0 was not installable for this Colab runtime; falling back to the latest supported 0.18.x wheel.")
        run(vllm_install_base + ["vllm>=0.14,<0.19"])

    # Torchvision imports Pillow during vLLM/Transformers initialization. Colab
    # images can end up with mixed PIL files after large dependency changes in a
    # live kernel, so force a coherent Pillow install before validation imports.
    run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "pillow>=10.0"])
    for module_name in list(sys.modules):
        if module_name == "PIL" or module_name.startswith("PIL."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", PLUGIN_SRC])

    # Editable installs write .pth metadata that is normally consumed at interpreter
    # startup. Make the source tree importable immediately in this live kernel.
    plugin_path = str(PLUGIN_SRC)
    if plugin_path not in sys.path:
        sys.path.insert(0, plugin_path)
    importlib.invalidate_caches()
    os.chdir(REPO_DIR / "notebooks")

    import torch
    import vllm
    import vllm_hook_plugins

    print(f"torch {torch.__version__} (CUDA runtime: {torch.version.cuda})")
    print(f"vLLM {vllm.__version__}")
    print(f"vllm_hook_plugins loaded from {Path(vllm_hook_plugins.__file__).parent}")
    print("Setup complete. Continue with the next cell; no runtime restart is needed.")
else:
    print("Not running in Colab; install dependencies from the repository README if needed.")


+ git -C /content/vLLM-Hook fetch origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
+ git -C /content/vLLM-Hook checkout main
Already on 'main'
Your branch is up to date with 'origin/main'.
+ git -C /content/vLLM-Hook pull --ff-only origin main
From https://github.com/IBM/vLLM-Hook
 * branch            main       -> FETCH_HEAD
Already up to date.
+ /usr/bin/python3 -m pip install -r /tmp/vllm_hook_colab_requirements.txt
+ /usr/bin/python3 -m pip install --upgrade protobuf>=5.29.6,<6.30
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.1
    Uninstalling protobuf-7.36.1:
      Successfully uninstalled protobuf-7.36.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.75.3 requires pro

### Imports & Environment


In [2]:
import io
import os
import multiprocessing as mp
import sys
from pathlib import Path

import torch
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


Environment configured


### Initialize `HookLLM`


In [3]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")


INFO 09-16 21:05:31 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/content/.cache/vllm-hook', 'dtype': torch.float16, 'max_model_len': 2048, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_extension_cls': 'vllm_hook_plugins.workers.spotlight_worker.SpotlightWorker', 'model': 'Qwen/Qwen2-1.5B-Instruct'}
WARNING 09-16 21:05:31 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

WARNING 09-16 21:05:41 [arg_utils.py:1390] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-16 21:06:01 [model.py:549] Resolved architecture: Qwen2ForCausalLM
WARNING 09-16 21:06:01 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 09-16 21:06:01 [model.py:1678] Using max model len 2048
WARNING 09-16 21:06:02 [arg_utils.py:2125] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.
INFO 09-16 21:06:02 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 09-16 21:06:02 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-16 21:06:02 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor 

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

INFO 09-16 21:06:03 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen2-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir='/content/.cache/vllm-hook', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detai

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

INFO 09-16 21:06:15 [weight_utils.py:581] Time spent downloading weights for Qwen/Qwen2-1.5B-Instruct: 6.636651 seconds
INFO 09-16 21:06:15 [weight_utils.py:625] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-16 21:06:16 [default_loader.py:384] Loading weights took 1.29 seconds
INFO 09-16 21:06:18 [gpu_model_runner.py:4820] Model loading took 2.89 GiB memory and 10.703396 seconds
INFO 09-16 21:06:21 [gpu_worker.py:436] Available KV cache memory: 36.49 GiB
INFO 09-16 21:06:21 [kv_cache_utils.py:1319] GPU KV cache size: 1,366,560 tokens
INFO 09-16 21:06:21 [kv_cache_utils.py:1324] Maximum concurrency for 2,048 tokens per request: 667.27x
INFO 09-16 21:06:21 [core.py:283] init engine (profile, create kv cache, warmup model) took 2.79 seconds
Model loaded: Qwen/Qwen2-1.5B-Instruct
Spotlight worker enabled


### Configure Test Parameters


In [4]:
prompt = (
    "Return the response for the following as a JSON: "
    "Write a 400 word paragraph on France and its food specifically "
    "focussing on dishes. Include the paragraph and the list of dishes "
    "mentioned the paragraph in seperate fields."
)
emph_strings = ["Return the response for the following as a JSON:"]
alpha = 0.1
temperature = 0.0
max_tokens = 500

sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens)


### Generate Baseline


In [5]:
outputs_baseline = llm.generate(
    prompts=[prompt],
    sampling_params=sampling_params,
    use_hook=False,
)
baseline_text = outputs_baseline[0].outputs[0].text
print(baseline_text)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 France is a country that is known for its rich culture, history, and cuisine. The French are known for their love of food and their ability to create delicious dishes that are both elegant and satisfying. One of the most famous dishes in France is the classic dish of escargot, which is a dish of snails cooked in garlic butter. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Another popular dish in France is the classic dish of coq au vin, which is a dish of chicken cooked in red wine and vegetables. Another popular dish in France is the classic dish of bouillabaisse, which is a dish of fish and seafood cooked in a broth. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Another popular dish in France is the classic dish of coq au vin, which is a dish of chicken cooked in red wine and vegetables. Another popular dish in France is the classic di

### Generate With Spotlight


In [6]:
outputs_spotlight = generate_with_spotlight(
    llm,
    prompts=[prompt],
    emph_strings=emph_strings,
    alpha=alpha,
    sampling_params=sampling_params,
)
spotlight_text = outputs_spotlight[0].outputs[0].text
print(spotlight_text)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{
  "paragraph": "France is a country known for its rich culinary traditions and diverse cuisine. From the classic French dishes like coq au vin and escargots to more modern creations like croissants and macarons, France has something for everyone. The country's cuisine is characterized by its use of fresh, high-quality ingredients and its emphasis on simplicity and balance. Some of the most famous French dishes include coq au vin, a dish made with chicken, wine, and mushrooms, and escargots, a dish made with snails cooked in garlic butter. Other popular French dishes include ratatouille, a vegetable stew made with tomatoes, zucchini, eggplant, and bell peppers, and bouillabaisse, a fish stew made with a variety of seafood and vegetables. In addition to these classic dishes, France also has a rich tradition of regional cuisine, with each region having its own unique dishes and specialties. From the hearty stews of the Auvergne region to the delicate pastries of the Loire Valley, France

### Comparison


In [7]:
print("=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"Prompt: {prompt}")
print(f"Emphasized span(s): {emph_strings}")
print(f"Alpha: {alpha}")
print("\nBASELINE")
print("-" * 70)
print(baseline_text)
print("\nWITH SPOTLIGHT")
print("-" * 70)
print(spotlight_text)


COMPARISON
Prompt: Return the response for the following as a JSON: Write a 400 word paragraph on France and its food specifically focussing on dishes. Include the paragraph and the list of dishes mentioned the paragraph in seperate fields.
Emphasized span(s): ['Return the response for the following as a JSON:']
Alpha: 0.1

BASELINE
----------------------------------------------------------------------
 France is a country that is known for its rich culture, history, and cuisine. The French are known for their love of food and their ability to create delicious dishes that are both elegant and satisfying. One of the most famous dishes in France is the classic dish of escargot, which is a dish of snails cooked in garlic butter. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Another popular dish in France is the classic dish of coq au vin, which is a dish of chicken cooked in red wine and vegetables. Another popular